# FTW Spain — Held-Out Test Inference

CUDA-based held-out inference for the final 3-epoch Mask R-CNN baseline using 216 independent test tiles.

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
!pip install -q geoai-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 650.8/650.8 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.2/669.2 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.4/811.4 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/

In [3]:
import geoai

print("GeoAI import: PASS")

GeoAI import: PASS


In [4]:
from pathlib import Path
import zipfile

zip_path = Path("/content/spain_test_images_uint8.zip")
extract_dir = Path("/content/spain_test")

extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

images_dir = extract_dir / "images_uint8"
images = sorted(images_dir.glob("*.tif"))

print("ZIP exists:", zip_path.exists())
print("Images directory:", images_dir)
print("Test images:", len(images))

if len(images) == 216:
    print("STATUS: PASS")
else:
    print("STATUS: FAIL")

ZIP exists: True
Images directory: /content/spain_test/images_uint8
Test images: 216
STATUS: PASS


In [5]:
from pathlib import Path
import rasterio
import numpy as np

model = Path("/content/best_model.pth")
images_dir = Path("/content/spain_test/images_uint8")
images = sorted(images_dir.glob("*.tif"))

print("=== FINAL INPUT QC ===")
print("Model exists:", model.exists())
print("Test images:", len(images))

bad = []

for path in images:
    with rasterio.open(path) as src:
        arr = src.read()

    if (
        arr.shape != (4, 256, 256)
        or arr.dtype != np.uint8
        or arr.min() < 0
        or arr.max() > 255
    ):
        bad.append(path.name)

print("Problems:", len(bad))

if model.exists() and len(images) == 216 and len(bad) == 0:
    print("STATUS: PASS")
else:
    print("STATUS: FAIL")

print("\nSample files:")
for path in [images[0], images[len(images)//2], images[-1]]:
    with rasterio.open(path) as src:
        arr = src.read()
    print(
        path.name,
        "| dtype:", arr.dtype,
        "| min:", arr.min(),
        "| max:", arr.max()
    )

=== FINAL INPUT QC ===
Model exists: True
Test images: 216
Problems: 0
STATUS: PASS

Sample files:
g1_00002_12.tif | dtype: uint8 | min: 15 | max: 255
g4_00021_2.tif | dtype: uint8 | min: 22 | max: 255
g7_00040_13.tif | dtype: uint8 | min: 28 | max: 255


In [6]:
from pathlib import Path
import time
import rasterio
import numpy as np
import geoai

model = Path("/content/best_model.pth")
images_dir = Path("/content/spain_test/images_uint8")
pred_dir = Path("/content/test_predictions")
pred_dir.mkdir(parents=True, exist_ok=True)

images = sorted(images_dir.glob("*.tif"))

print("=== SPAIN HELD-OUT CUDA INFERENCE ===")
print("Images:", len(images))
print("Device: CUDA")
print("Threshold: 0.3")
print()

start = time.time()

empty_predictions = 0
total_instances = 0
failed = []

for i, img in enumerate(images, start=1):

    pred = pred_dir / f"{img.stem}_pred.tif"

    try:
        result = geoai.instance_segmentation(
            input_path=str(img),
            output_path=str(pred),
            model_path=str(model),
            num_channels=4,
            num_classes=2,
            confidence_threshold=0.3,
            device="cuda",
        )

        with rasterio.open(pred) as src:
            arr = src.read(1)

        instances = len(np.unique(arr)) - 1
        nonzero = np.count_nonzero(arr)

        total_instances += instances

        if nonzero == 0:
            empty_predictions += 1

    except Exception as e:
        failed.append((img.name, str(e)))

    if i % 25 == 0 or i == len(images):
        elapsed = time.time() - start
        print(
            f"{i:3d}/{len(images)} | "
            f"empty={empty_predictions} | "
            f"instances={total_instances} | "
            f"elapsed={elapsed/60:.1f} min"
        )

elapsed = time.time() - start

print("\n=== INFERENCE COMPLETE ===")
print("Processed:", len(images))
print("Prediction files:", len(list(pred_dir.glob("*.tif"))))
print("Empty predictions:", empty_predictions)
print("Total instances:", total_instances)
print("Failed:", len(failed))
print("Total time:", round(elapsed / 60, 2), "min")
print("Mean time/tile:", round(elapsed / len(images), 2), "sec")

if failed:
    print("\nFailed files:")
    for name, error in failed[:10]:
        print("-", name, ":", error)

=== SPAIN HELD-OUT CUDA INFERENCE ===
Images: 216
Device: CUDA
Threshold: 0.3

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:01<00:00, 114MB/s]


 25/216 | empty=1 | instances=613 | elapsed=0.9 min


 50/216 | empty=2 | instances=939 | elapsed=1.4 min


 75/216 | empty=2 | instances=1420 | elapsed=2.0 min


...

125/216 | empty=2 | instances=2236 | elapsed=3.1 min


150/216 | empty=14 | instances=2401 | elapsed=3.6 min


175/216 | empty=22 | instances=2554 | elapsed=4.1 min


200/216 | empty=26 | instances=2808 | elapsed=4.6 min


216/216 | empty=27 | instances=3143 | elapsed=5.0 min

=== INFERENCE COMPLETE ===
Processed: 216
Prediction files: 648
Empty predictions: 27
Total instances: 3143
Failed: 0
Total time: 4.96 min
Mean time/tile: 1.38 sec


In [7]:
from pathlib import Path
import zipfile

pred_dir = Path("/content/test_predictions")
zip_path = Path("/content/spain_test_predictions_cuda.zip")

files = sorted(pred_dir.glob("*.tif"))

print("Prediction TIFF files:", len(files))

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in files:
        zf.write(file, arcname=file.name)

print("ZIP created:", zip_path)
print("Size MB:", round(zip_path.stat().st_size / 1024**2, 2))

Prediction TIFF files: 648
ZIP created: /content/spain_test_predictions_cuda.zip
Size MB: 1.54
